# 07 · Sessions (FIFO + state per session)

A **session-enabled** queue (or subscription) guarantees:

- **FIFO ordering** *within* a `SessionId`
- **Single consumer** per session at a time (the broker locks the whole session)
- Optional **session state** — a small blob stored against the session

Sessions are how you partition a stream of messages by customer / order / chat-id while preserving order **within** each partition.

We deployed `demo-sessions` with `requiresSession = true`.


In [ ]:
#r "nuget: Azure.Messaging.ServiceBus, 7.18.2"

In [ ]:
#!import ../src/shared/Config.cs

In [ ]:
using Azure.Messaging.ServiceBus;

var client = new ServiceBusClient(Config.ConnectionString);
var sender = client.CreateSender(Config.SessionQueueName);

## 1. Send messages for two different sessions, interleaved

In [ ]:
var messages = new[]
{
    new ServiceBusMessage("A1") { SessionId = "alice" },
    new ServiceBusMessage("B1") { SessionId = "bob"   },
    new ServiceBusMessage("A2") { SessionId = "alice" },
    new ServiceBusMessage("B2") { SessionId = "bob"   },
    new ServiceBusMessage("A3") { SessionId = "alice" },
    new ServiceBusMessage("B3") { SessionId = "bob"   },
};
await sender.SendMessagesAsync(messages);
Console.WriteLine("Sent 3 messages each for alice and bob.");

## 2. Accept a session and drain it in order

`AcceptNextSessionAsync()` gives you the *next available* session; `AcceptSessionAsync("alice")` grabs a specific one.

In [ ]:
var sessionReceiver = await client.AcceptNextSessionAsync(Config.SessionQueueName);
Console.WriteLine($"Locked session: {sessionReceiver.SessionId}");

while (true)
{
    var msg = await sessionReceiver.ReceiveMessageAsync(TimeSpan.FromSeconds(2));
    if (msg is null) break;
    Console.WriteLine($"  [{sessionReceiver.SessionId}] {msg.Body}");
    await sessionReceiver.CompleteMessageAsync(msg);
}
await sessionReceiver.CloseAsync();

## 3. Session state (per-session scratch space)

In [ ]:
var sr = await client.AcceptNextSessionAsync(Config.SessionQueueName);
Console.WriteLine($"Locked session: {sr.SessionId}");

await sr.SetSessionStateAsync(new BinaryData($"processed-up-to:{DateTimeOffset.UtcNow:o}"));
var state = await sr.GetSessionStateAsync();
Console.WriteLine($"State: {state}");

while (true)
{
    var msg = await sr.ReceiveMessageAsync(TimeSpan.FromSeconds(2));
    if (msg is null) break;
    Console.WriteLine($"  [{sr.SessionId}] {msg.Body}");
    await sr.CompleteMessageAsync(msg);
}
await sr.CloseAsync();

## 4. `ServiceBusSessionProcessor` — the high-level API

Like `ServiceBusProcessor` but session-aware. It locks one session per handler-thread, drains it, then moves to the next.

In [ ]:
// Seed again so we have work
await sender.SendMessagesAsync(new[]
{
    new ServiceBusMessage("X1") { SessionId = "carol" },
    new ServiceBusMessage("X2") { SessionId = "carol" },
    new ServiceBusMessage("Y1") { SessionId = "dave"  },
});

var sp = client.CreateSessionProcessor(Config.SessionQueueName, new ServiceBusSessionProcessorOptions
{
    MaxConcurrentSessions = 2,
    AutoCompleteMessages = true,
});

sp.ProcessMessageAsync += async args =>
{
    Console.WriteLine($"[session {args.SessionId}] {args.Message.Body}");
    await Task.CompletedTask;
};
sp.ProcessErrorAsync += args => { Console.WriteLine(args.Exception.Message); return Task.CompletedTask; };

await sp.StartProcessingAsync();
await Task.Delay(TimeSpan.FromSeconds(4));
await sp.StopProcessingAsync();
await sp.DisposeAsync();

await sender.DisposeAsync();
await client.DisposeAsync();

Next: [`08-topics-subscriptions.ipynb`](08-topics-subscriptions.ipynb)